# Use the RDF Data of the Swiss Political Parties

In [1]:
# load graph from the Turtle file
from rdflib import Graph

g = Graph()
g.parse("../data/output/parties.ttl", format="turtle")

<Graph identifier=N02362ce08795420485f4d5084b97990b (<class 'rdflib.graph.Graph'>)>

In [2]:
import pandas as pd

def sparql_to_df(graph: Graph, query: str) -> pd.DataFrame:
    result = graph.query(query)
    return pd.DataFrame(
        [
            [val.toPython() if val is not None else None for val in row]
            for row in result
        ],
        columns=[str(var) for var in result.vars]
    )

## Number of Versions per Party

In [3]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?party (COUNT(?version) AS ?count) WHERE {
    ?party a vl:Identity .
    ?version vl:identity ?party .
} GROUP BY ?party ORDER BY DESC(?count)

"""

In [4]:
df = sparql_to_df(g, query)

display(df)

,party,count
0,https://politics.ld.admin.ch/party-id/128,11
1,https://politics.ld.admin.ch/party-id/311,8
2,https://politics.ld.admin.ch/party-id/37,8
3,https://politics.ld.admin.ch/party-id/103,7
4,https://politics.ld.admin.ch/party-id/196,7
...,...,...
301,https://politics.ld.admin.ch/party-id/91,1
302,https://politics.ld.admin.ch/party-id/95,1
303,https://politics.ld.admin.ch/party-id/96,1
304,https://politics.ld.admin.ch/party-id/98,1


## Identities without Versions

Should not happen

In [5]:
query = """

PREFIX vl: <https://version.link/>

SELECT ?id WHERE {
    ?id a vl:Identity .
    FILTER NOT EXISTS {
        ?id vl:version ?version .
    }
}
    
"""

In [6]:
df = sparql_to_df(g, query)

display(df)

,id
0,https://politics.ld.admin.ch/party-id/64
1,https://politics.ld.admin.ch/party-id/93


## All Relationships of a Party Version

In [7]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?version ?from ?name WHERE {
    <https://politics.ld.admin.ch/party-version/1_1971-09-22> (vl:successor|vl:predecessor)+ ?version .

    ?version schema:validFrom ?from .

    OPTIONAL {
        ?version schema:name ?name .
        FILTER (lang(?name) = "de")
    }

} ORDER BY ?from
    
"""

In [8]:
df = sparql_to_df(g, query)

display(df)

,version,from,name
0,https://politics.ld.admin.ch/party-version/16_...,1880-01-01,Vorläufer Demokraten
1,https://politics.ld.admin.ch/party-version/17_...,1905-03-19,Schweizerische Demokratische Partei
2,https://politics.ld.admin.ch/party-version/10_...,1912-04-22,Schweizerische Konservative Volkspartei
3,https://politics.ld.admin.ch/party-version/209...,1920-01-01,"Vorläufer Bauern-, Gewerbe und Bürgerpartei"
4,https://politics.ld.admin.ch/party-version/11_...,1936-12-23,"Bauern-, Gewerbe- und Bürgerpartei"
5,https://politics.ld.admin.ch/party-version/18_...,1941-07-01,Demokratische Partei der Schweiz
6,https://politics.ld.admin.ch/party-version/10_...,1957-07-01,Konservativ-Christlichsoziale Volkspartei
7,https://politics.ld.admin.ch/party-version/10_...,1970-12-12,Christlichdemokratische Volkspartei der Schweiz
8,https://politics.ld.admin.ch/party-version/1_1...,1971-09-22,Schweizerische Volkspartei
9,https://politics.ld.admin.ch/party-version/65_...,1976-05-22,Christlichsoziale Partei der Schweiz


## Visualization of the Relationships of a Party Version

In [13]:
query = """

PREFIX vl: <https://version.link/>
PREFIX schema: <http://schema.org/>

SELECT ?source ?target ?source_name ?target_name WHERE {
    
    <https://politics.ld.admin.ch/party-version/43_1983-06-15> (vl:successor|vl:predecessor)+ ?source .

    ?source vl:successor ?target .
        
    OPTIONAL {
        ?source schema:name ?source_name .
        FILTER (lang(?source_name) = "de")
    }
    OPTIONAL {
        ?target schema:name ?target_name .
        FILTER (lang(?target_name) = "de")
    }
}
    
"""

In [14]:
df = sparql_to_df(g, query)

display(df)

,source,target,source_name,target_name
0,https://politics.ld.admin.ch/party-version/43_...,https://politics.ld.admin.ch/party-version/43_...,Grüne Alternative Schweiz,None


In [12]:
from ipycytoscape import CytoscapeWidget
import networkx as nx

my_style = [
    {
        "selector": "node",
        "style": {
            "label": "data(name)",
            "background-color": "#0074D9",
            "color": "red",
            "text-valign": "center",
            "text-halign": "center",
            "font-size": 10,
            "width": 20,
            "height": 20,
        },
    },
    {
        "selector": "edge.directed",
        "style": {
            "line-color": "#0074D9",
            "target-arrow-color": "#0074D9",
            "target-arrow-shape": "triangle",
            "curve-style": "bezier",
        },
    },
]

ids = pd.Series.tolist(df['source']) + pd.Series.tolist(df['target'])
names = pd.Series.tolist(df['source_name']) + pd.Series.tolist(df['target_name'])

node_df = pd.DataFrame({'id': ids, 'name': names})
node_df = node_df.drop_duplicates(subset=['id']).reset_index(drop=True)

G = nx.from_pandas_edgelist(df, source='source', target='target', create_using=nx.DiGraph())
nx.set_node_attributes(G, node_df.set_index('id').to_dict('index'))

cyto = CytoscapeWidget()
cyto.graph.add_graph_from_networkx(G, directed=True)
cyto.set_style(my_style)
cyto.set_layout(name = "dagre", rankDir = "LR")

cyto

/Users/hib1/Nextcloud BFH/Projekte/26_BK_Parteienatlas/rdf-parties/.venv/lib/python3.13/site-packages/jupyter_client/session.py:727: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'LR'}, cytoscape_style=[{'selector': 'node', 'st…